# Fast convolution

So, the english is not very good, I will improve that

In [1]:
import itertools

import sympy as sy
import numpy as np

In [2]:
import fitz
from IPython.display import YouTubeVideo

In [3]:
from utils import plot_pdf

In [4]:
doc = fitz.open('Blahut_2010_Fast algorithms for signal processing.pdf')

The base for this tutorial is the book "Fast Algorithms or Signal Processing" of Blahut.
Is an excellent book but full of examples but, of course, do not explain everything in details.
For that parts I will quote another books and videos from YouTube.

The book puts first all theory and only after that we have a detailed example. To better understanding in this tutorial we have quotes from theory and examples together fallowing the sequence of commands

This example will not work for different vector sizes.

## Start

Size of vectors

In [5]:
d_num = 4
g_num = 3

In [6]:
b_degree = d_num + g_num - 1
b_degree

6

In [7]:
# bi = [1, 2, -2, sy.Rational(1, 2)]
bi = [0, 1, -1, -2, 2, sy.oo]
sy.Matrix(bi)

Matrix([
[ 0],
[ 1],
[-1],
[-2],
[ 2],
[oo]])

Example of vectors for the convolution

In [8]:
#d_values = [56789012, 78901234, 123456] 
#g_values = [54321098, 43219876, 98765]


In [9]:
d_values = list(range(1, d_num+1))
g_values = list(range(1, g_num+1))
print(d_values, g_values)

[1, 2, 3, 4] [1, 2, 3]


Polynomial degree

In [10]:
d_degree = d_num - 1
g_degree = g_num - 1
print(d_degree, g_degree)

3 2


In [11]:
di = sy.Matrix(sy.symbols(" ".join(f"d_{i}"for i in range(d_num))))
di

Matrix([
[d_0],
[d_1],
[d_2],
[d_3]])

In [12]:
gi = sy.Matrix(sy.symbols(" ".join(f"g_{i}"for i in range(g_num))))
gi

Matrix([
[g_0],
[g_1],
[g_2]])

In [13]:
_a_mtx = [[(b **e) for e, d in enumerate(di)] for b in bi if b != sy.oo]
_a_inf = [[0] * (len(di) - 1) + [1]]
a_mtx = sy.Matrix(_a_mtx + _a_inf)
a_mtx

Matrix([
[1,  0, 0,  0],
[1,  1, 1,  1],
[1, -1, 1, -1],
[1, -2, 4, -8],
[1,  2, 4,  8],
[0,  0, 0,  1]])

In [14]:
_b_mtx = [[(b **e) for e, d in enumerate(gi)] for b in bi if b != sy.oo]
_b_inf = [[0] * (len(gi) - 1) + [1]]
b_mtx = sy.Matrix(_b_mtx + _b_inf)
b_mtx

Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[1, -2, 4],
[1,  2, 4],
[0,  0, 1]])

In [15]:
bg_mtx = sy.diag(*(b_mtx * gi).tolist())
bg_mtx

Matrix([
[g_0,               0,               0,                   0,                   0,   0],
[  0, g_0 + g_1 + g_2,               0,                   0,                   0,   0],
[  0,               0, g_0 - g_1 + g_2,                   0,                   0,   0],
[  0,               0,               0, g_0 - 2*g_1 + 4*g_2,                   0,   0],
[  0,               0,               0,                   0, g_0 + 2*g_1 + 4*g_2,   0],
[  0,               0,               0,                   0,                   0, g_2]])

In [16]:
_c_mtx = [[(b **e) for e in range(b_degree)] for b in bi if b != sy.oo]
_c_inf = [[0] * (b_degree - 1) + [1]]
c_mtx = sy.Matrix(_c_mtx + _c_inf)
c_mtx

Matrix([
[1,  0, 0,  0,  0,   0],
[1,  1, 1,  1,  1,   1],
[1, -1, 1, -1,  1,  -1],
[1, -2, 4, -8, 16, -32],
[1,  2, 4,  8, 16,  32],
[0,  0, 0,  0,  0,   1]])

In [17]:
c_inv0 = c_mtx.inv()
c_inv0

Matrix([
[   1,    0,    0,     0,     0,  0],
[   0,  2/3, -2/3,  1/12, -1/12,  4],
[-5/4,  2/3,  2/3, -1/24, -1/24,  0],
[   0, -1/6,  1/6, -1/12,  1/12, -5],
[ 1/4, -1/6, -1/6,  1/24,  1/24,  0],
[   0,    0,    0,     0,     0,  1]])

In [18]:
s = sy.MatMul(c_inv0, bg_mtx, a_mtx, sy.Matrix(di))
s

Matrix([
[   1,    0,    0,     0,     0,  0],
[   0,  2/3, -2/3,  1/12, -1/12,  4],
[-5/4,  2/3,  2/3, -1/24, -1/24,  0],
[   0, -1/6,  1/6, -1/12,  1/12, -5],
[ 1/4, -1/6, -1/6,  1/24,  1/24,  0],
[   0,    0,    0,     0,     0,  1]])*Matrix([
[g_0,               0,               0,                   0,                   0,   0],
[  0, g_0 + g_1 + g_2,               0,                   0,                   0,   0],
[  0,               0, g_0 - g_1 + g_2,                   0,                   0,   0],
[  0,               0,               0, g_0 - 2*g_1 + 4*g_2,                   0,   0],
[  0,               0,               0,                   0, g_0 + 2*g_1 + 4*g_2,   0],
[  0,               0,               0,                   0,                   0, g_2]])*Matrix([
[1,  0, 0,  0],
[1,  1, 1,  1],
[1, -1, 1, -1],
[1, -2, 4, -8],
[1,  2, 4,  8],
[0,  0, 0,  1]])*Matrix([
[d_0],
[d_1],
[d_2],
[d_3]])

## Example

In [20]:
subs = {k[0]: v for k, v in zip(di.tolist()+gi.tolist(), d_values + g_values)}
subs

{d_0: 1, d_1: 2, d_2: 3, d_3: 4, g_0: 1, g_1: 2, g_2: 3}

In [21]:
si = s.subs(subs)
si

Matrix([
[   1,    0,    0,     0,     0,  0],
[   0,  2/3, -2/3,  1/12, -1/12,  4],
[-5/4,  2/3,  2/3, -1/24, -1/24,  0],
[   0, -1/6,  1/6, -1/12,  1/12, -5],
[ 1/4, -1/6, -1/6,  1/24,  1/24,  0],
[   0,    0,    0,     0,     0,  1]])*Matrix([
[1, 0, 0, 0,  0, 0],
[0, 6, 0, 0,  0, 0],
[0, 0, 2, 0,  0, 0],
[0, 0, 0, 9,  0, 0],
[0, 0, 0, 0, 17, 0],
[0, 0, 0, 0,  0, 3]])*Matrix([
[1,  0, 0,  0],
[1,  1, 1,  1],
[1, -1, 1, -1],
[1, -2, 4, -8],
[1,  2, 4,  8],
[0,  0, 0,  1]])*Matrix([
[1],
[2],
[3],
[4]])

Let's compare the output polynomial matrix from direct and winograd method

In [22]:
sy.Matrix(np.convolve(np.array(di).reshape(-1), np.array(gi).reshape(-1)))

Matrix([
[                    d_0*g_0],
[          d_0*g_1 + d_1*g_0],
[d_0*g_2 + d_1*g_1 + d_2*g_0],
[d_1*g_2 + d_2*g_1 + d_3*g_0],
[          d_2*g_2 + d_3*g_1],
[                    d_3*g_2]])

In [23]:
se = sy.MatMul(c_inv0, bg_mtx, a_mtx, sy.Matrix(di), evaluate=True)
se

Matrix([
[                    d_0*g_0],
[          d_0*g_1 + d_1*g_0],
[d_0*g_2 + d_1*g_1 + d_2*g_0],
[d_1*g_2 + d_2*g_1 + d_3*g_0],
[          d_2*g_2 + d_3*g_1],
[                    d_3*g_2]])

Comparing numerical outputs from direct and winograd method

In [24]:
sy.Matrix(np.convolve(d_values, g_values))

Matrix([
[ 1],
[ 4],
[10],
[16],
[17],
[12]])

In [25]:
se.subs(subs)

Matrix([
[ 1],
[ 4],
[10],
[16],
[17],
[12]])